# Stage 5 - Gaussian process: a map of the prices the model does not explain

**Input:** `data/processed/sales_features.csv` and `data/processed/boosting_predictions.csv` (from `04_models.ipynb`)
**Output:** `data/processed/gp_surface.csv` (the map) and `data/processed/gp_by_distance.csv`

Stage 4 measured the border effect by giving the model a distance. This stage checks it another way, **without telling the model where the border is**.

1. Take the gradient boosting *without any border variable*. Its error on each sale (actual price minus predicted price) is what the home and its municipality do not explain.
2. Fit a Gaussian process on these errors, using only the position of the sale. A Gaussian process assumes that nearby sales have similar errors and learns how fast this similarity fades with distance. The result is a smooth map.
3. If the border matters, the map should show prices above the prediction along the Geneva border, and this should disappear once the distance to Geneva is given to the model.

A Gaussian process becomes very slow with many points, so it is fitted on a random sample of 2,000 sales.

*Tool declaration: this notebook was drafted with Claude (Anthropic), asked to fit a Gaussian process on the residuals of the stage 4 model and to map it.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.metrics import r2_score
from sklearn.neighbors import NearestNeighbors

from helpers import PROCESSED, SEED, load_border_crossings, load_model_data, to_km

SAMPLE_SIZE = 2_000     # sales used to fit the Gaussian process
CHECK_SIZE = 10_000     # other sales used to check it
GRID_STEP_KM = 1.0      # size of the cells of the map
MAX_GAP_KM = 2.0        # a cell is drawn only if a sale exists within this distance

sales = load_model_data().merge(pd.read_csv(PROCESSED / "boosting_predictions.csv"),
                                on="id_mutation", validate="one_to_one")
sales["residual_no_border"] = sales["price_vs_quarter"] - sales["prediction_no_border"]
sales["residual_geneva"] = sales["price_vs_quarter"] - sales["prediction_geneva"]
sales["x_km"], sales["y_km"] = to_km(sales["latitude"], sales["longitude"])
print(f"{len(sales):,} sales")

## 1. Sample and fit

The kernel has three parts, each with a plain meaning:

- `ConstantKernel`: how large the local price differences are;
- `RBF`: over how many kilometres two places still have similar prices (the length scale);
- `WhiteKernel`: the part of each sale that no location can explain (condition of the home, floor, view, negotiation).

The three numbers are learned from the data.

In [ ]:
rng = np.random.default_rng(SEED)
shuffled = rng.permutation(len(sales))
sample = sales.iloc[shuffled[:SAMPLE_SIZE]]
check = sales.iloc[shuffled[SAMPLE_SIZE:SAMPLE_SIZE + CHECK_SIZE]]

def fit_gaussian_process(residual_column):
    """Fit a Gaussian process of the given residual on the position (x_km, y_km) of the sampled sales."""
    kernel = ConstantKernel(0.05) * RBF(length_scale=5.0, length_scale_bounds=(0.5, 50)) + WhiteKernel(0.1)
    gp = GaussianProcessRegressor(kernel, random_state=SEED)
    return gp.fit(sample[["x_km", "y_km"]], sample[residual_column])

gp_no_border = fit_gaussian_process("residual_no_border")
print("Learned kernel:", gp_no_border.kernel_)

## 2. Does the map predict anything?

Check on 10,000 sales that were not in the sample: the share of the residual explained by the map alone. Note that these sales come from the same municipalities as the sample, so this is a test of interpolation between known places, not of unseen municipalities as in stage 4.

In [ ]:
check_prediction = gp_no_border.predict(check[["x_km", "y_km"]])
print(f"R2 of the map on {len(check):,} other sales: {r2_score(check['residual_no_border'], check_prediction):.3f}")

## 3. The map

The Gaussian process is evaluated on a grid of 1 km cells. Cells with no sale within 2 km (mountains, lake, Switzerland) are left empty: the model has nothing to say there.

In [ ]:
xs = np.arange(sales["x_km"].min(), sales["x_km"].max() + GRID_STEP_KM, GRID_STEP_KM)
ys = np.arange(sales["y_km"].min(), sales["y_km"].max() + GRID_STEP_KM, GRID_STEP_KM)
grid = pd.DataFrame([(x, y) for x in xs for y in ys], columns=["x_km", "y_km"])

distance_to_sale, _ = NearestNeighbors(n_neighbors=1).fit(sales[["x_km", "y_km"]]).kneighbors(grid)
grid = grid[distance_to_sale[:, 0] <= MAX_GAP_KM].reset_index(drop=True)

grid["surface"], grid["uncertainty"] = gp_no_border.predict(grid, return_std=True)
grid["surface_pct"] = 100 * (np.exp(grid["surface"]) - 1)
grid.to_csv(PROCESSED / "gp_surface.csv", index=False)
print(f"{len(grid):,} cells drawn")

In [ ]:
crossings = load_border_crossings()
crossings["x_km"], crossings["y_km"] = to_km(crossings["lat"], crossings["lon"])
geneva = crossings[crossings["canton"] == "GE"]
valais = crossings[crossings["canton"] == "VS"]

fig, ax = plt.subplots(figsize=(9, 6.5))
limit = np.abs(grid["surface_pct"]).quantile(0.99)
cells = ax.scatter(grid["x_km"], grid["y_km"], c=grid["surface_pct"], cmap="RdBu_r",
                   vmin=-limit, vmax=limit, s=14, marker="s")
ax.scatter(geneva["x_km"], geneva["y_km"], color="black", marker="x", s=50, label="Crossings into Geneva")
ax.scatter(valais["x_km"], valais["y_km"], color="black", marker="^", s=40, label="Crossings into Valais")
fig.colorbar(cells, label="Price vs. prediction of the model without border (%)")
ax.set_xlabel("Kilometres east of the reference point")
ax.set_ylabel("Kilometres north of the reference point")
ax.set_title("Local price differences not explained by the home and its municipality")
ax.set_aspect("equal")
ax.legend(loc="lower left")
plt.show()

## 4. Is the unexplained price higher near Geneva?

Average value of the map by distance to the nearest Geneva crossing, for two versions:

- residuals of the model **without** border variable: the map is free to show a border effect;
- residuals of the model **with** the distance to Geneva: if the distance already captures the effect, nothing should be left near the border.

The averages are taken over the 10,000 check sales, so that places with many sales count more.

In [ ]:
gp_geneva = fit_gaussian_process("residual_geneva")

check = check.assign(
    map_no_border=check_prediction,
    map_geneva=gp_geneva.predict(check[["x_km", "y_km"]]),
)
bands = pd.cut(check["dist_geneva_km"], [0, 2, 5, 10, 20, 30, 60])
by_distance = check.groupby(bands, observed=False).agg(
    sales=("map_no_border", "size"),
    map_no_border_pct=("map_no_border", lambda v: 100 * (np.exp(v.mean()) - 1)),
    map_with_geneva_distance_pct=("map_geneva", lambda v: 100 * (np.exp(v.mean()) - 1)),
)
by_distance.to_csv(PROCESSED / "gp_by_distance.csv")
by_distance.round(1)